# 3. SQL Data Ingestion

This notebook loads the clustered EMR data into PostgreSQL for Vanna to query.

In [1]:
import os
import sys
import pandas as pd
from sqlalchemy import create_engine

# Ultimate foolproof path injection to find 'src' regardless of current working directory
cwd = os.path.abspath(os.getcwd())
project_root = None

# 1. Walk up from current working directory
temp = cwd
for _ in range(4):
    if os.path.exists(os.path.join(temp, 'src', 'config.py')):
        project_root = temp
        break
    parent = os.path.abspath(os.path.join(temp, '..'))
    if parent == temp:
        break
    temp = parent

# 2. Fallback to searching sys.path
if not project_root:
    for path in list(sys.path):
        if not path:
            continue
        candidate = os.path.abspath(path)
        for folder in [candidate, os.path.abspath(os.path.join(candidate, '..')), os.path.abspath(os.path.join(candidate, '..', '..'))]:
            if os.path.exists(os.path.join(folder, 'src', 'config.py')):
                project_root = folder
                break
        if project_root:
            break

if project_root:
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
    path_prefix = os.path.relpath(project_root, cwd)
    print(f"Project root found: {project_root}")
    print(f"Current working directory: {cwd}")
    print(f"Relative path prefix: {path_prefix}")
else:
    print("Warning: Could not automatically detect project root. Using default path prefix '..'")
    path_prefix = ".."

from src.config import settings

Project root found: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator
Current working directory: d:\PEMROGRAMAN\LLM-DESKTOP\local-rag-comparator\notebook
Relative path prefix: ..


In [2]:
print("1. Loading Clustered Data...")
file_path = os.path.join(path_prefix, "output", "clustered_emr.csv")
if not os.path.exists(file_path):
    raise FileNotFoundError("clustered_emr.csv not found. Run 1_clustering.ipynb first.")

df = pd.read_csv(file_path)
print(f"Loaded {len(df)} records.")

1. Loading Clustered Data...
Loaded 20630 records.


In [3]:
print("2. Cleaning and Formatting Data for SQL...")

# Clean column names to be SQL-friendly (lowercase, underscores)
df.columns = [
    c.strip().lower().replace(' / ', '_').replace(' ', '_').replace('/', '_') 
    for c in df.columns
]

# Ensure model_family column is created dynamically
if 'model_family' not in df.columns and 'machine_model' in df.columns:
    print("  Creating 'model_family' column from 'machine_model'...")
    df['model_family'] = df['machine_model'].apply(lambda x: str(x).split('-')[0] if pd.notna(x) else x)

# Ensure date columns are proper datetime objects
date_cols = ['created_date', 'closed_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Drop combined_text as we don't need it in the structured DB
if 'combined_text' in df.columns:
    df = df.drop(columns=['combined_text'])

print("Columns ready for SQL:", df.columns.tolist())

2. Cleaning and Formatting Data for SQL...
  Creating 'model_family' column from 'machine_model'...
Columns ready for SQL: ['status', 'emr_name', 'serial_number', 'machine_product', 'machine_model', 'account:_account_name', 'sub_call_type', 'pmact_type', 'subjects', 'symptom', 'caused_of_problem', 'action_(how_was_problem_corrected?)', 'branch_site', 'part_suply', 'main_cause_part_no', 'part_description', 'part_description.1', 'techcare_component', 'techcare_component.1', 'techcare_sub_component', 'techcare_sub_component.1', 'created_date', 'emr_last_closed_date', 'cluster_id', 'cluster_label', 'cluster_confidence', 'action_cluster_id', 'action_canonical', 'symptom_cluster_id', 'symptom_canonical', 'cause_canonical', 'model_family']


In [4]:
print("3. Connecting to PostgreSQL and Ingesting Data...")
engine = create_engine(settings.postgres_url)

table_name = 'emr_records'
print(f"Writing to table '{table_name}'...")

# Write data to postgres
df.to_sql(table_name, engine, if_exists='replace', index=False)
print("Ingestion complete.")

3. Connecting to PostgreSQL and Ingesting Data...
Writing to table 'emr_records'...
Ingestion complete.


In [5]:
print("4. Testing SQL Query...")
test_query = f"SELECT machine_model, COUNT(*) as fault_count FROM {table_name} GROUP BY machine_model ORDER BY fault_count DESC LIMIT 5"
result = pd.read_sql(test_query, engine)
print(result)

4. Testing SQL Query...
  machine_model  fault_count
0       HD785-7         3586
1   PC135F-10M0         2207
2     D85E-SS-2         1286
3    PC200-10M0          827
4       D155A-6          758
